In [15]:
import pandas as pd

In [16]:
clean_inventario_demanda_df = pd.read_parquet('../data/processed/inventario_demanda_limpio.parquet')
clean_inventario_demanda_df

,fecha,id_producto,nombre_producto,categoria,id_sucursal,nombre_sucursal,proveedor,unidades_vendidas,unidades_recibidas,stock_cierre,precio_unitario,costo_unitario,importe_venta,importe_costo
0,2024-06-06,P025,Papel Higiénico 4-pack,Higiene,S01,Mérida Centro,Grupo Abasto del Sur,45,0,56,61.01,30.0,2745.45,1350.0
1,2023-08-17,P007,Jabón Multiusos 500ml,Limpieza,S05,Playa del Carmen,Proveedor Central del Caribe,9,0,146,41.74,19.0,375.66,171.0
2,2023-12-14,P018,Café Molido 250g,Abarrotes,S04,Cancún Centro,Comercializadora Peninsular,14,0,203,84.89,42.0,1188.46,588.0
3,2023-07-09,P006,Detergente Líquido 1L,Limpieza,S04,Cancún Centro,Logística Maya S.A.,16,0,218,72.62,35.0,1161.92,560.0
4,2023-11-25,P010,Esponja Cocina Pack 3,Limpieza,S01,Mérida Centro,Comercializadora Peninsular,13,95,278,33.61,15.0,436.93,195.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52734,2023-11-17,P006,Detergente Líquido 1L,Limpieza,S02,Mérida Norte,N/A,8,0,142,75.26,35.0,602.08,280.0
52735,2024-06-28,P025,Papel Higiénico 4-pack,Higiene,S03,Cancún,Distribuidora Norte S.A.,23,175,175,63.07,30.0,1450.61,690.0
52736,2023-12-04,P021,Queso Manchego 400g,Lácteos,S02,Mérida Norte,Grupo Abasto del Sur,8,188,188,92.44,50.0,739.52,400.0
52737,2024-03-02,P001,Agua Mineral 1L,Bebidas,S02,Mérida Norte,Comercializadora Peninsular,23,0,131,18.69,8.0,429.87,184.0


In [17]:
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

eda_df = (
    clean_inventario_demanda_df
    .loc[lambda d: d["fecha"].dt.year == 2023]
    .assign(
        dia_semana=lambda d: pd.Categorical(d["fecha"].dt.dayofweek.map(dict(enumerate(dias))), categories=dias, ordered=True),
        mes=lambda d: pd.Categorical(d["fecha"].dt.month.map(dict(enumerate(meses, start=1))), categories=meses, ordered=True),
    )
    [["fecha", "dia_semana", "mes", "nombre_sucursal", "nombre_producto", "unidades_vendidas"]]
)

eda_df

,fecha,dia_semana,mes,nombre_sucursal,nombre_producto,unidades_vendidas
1,2023-08-17,Jueves,Agosto,Playa del Carmen,Jabón Multiusos 500ml,9
2,2023-12-14,Jueves,Diciembre,Cancún Centro,Café Molido 250g,14
3,2023-07-09,Domingo,Julio,Cancún Centro,Detergente Líquido 1L,16
4,2023-11-25,Sábado,Noviembre,Mérida Centro,Esponja Cocina Pack 3,13
5,2023-05-27,Sábado,Mayo,Cancún,Aceite Vegetal 1L,10
...,...,...,...,...,...,...
52732,2023-06-16,Viernes,Junio,Mérida Norte,Esponja Cocina Pack 3,16
52733,2023-03-25,Sábado,Marzo,Playa del Carmen,Jugo Naranja 1L,14
52734,2023-11-17,Viernes,Noviembre,Mérida Norte,Detergente Líquido 1L,8
52736,2023-12-04,Lunes,Diciembre,Mérida Norte,Queso Manchego 400g,8


In [22]:
# Producto × sucursal
por_sucursal = eda_df.pivot_table(
    index="nombre_producto", columns="nombre_sucursal",
    values="unidades_vendidas", aggfunc="mean", observed=True, fill_value=0,
)
por_sucursal

nombre_sucursal,Cancún,Cancún Centro,Mérida Centro,Mérida Norte,Playa del Carmen
nombre_producto,,,,,
Aceite Vegetal 1L,13.864407,13.643154,13.343096,14.113043,13.872340
Agua Mineral 1L,40.932722,40.506410,40.566879,39.795031,40.184375
Agua Sabor Limón 600ml,15.573276,15.179825,15.066038,15.331915,15.855140
Arroz Largo 1kg,39.241935,37.147799,38.506211,39.176656,38.775000
Atún Lata 140g,39.399381,38.383686,36.922840,38.327974,37.009404
Azúcar Blanca 1kg,14.148837,14.468468,14.118421,14.073394,13.873077
Café Molido 250g,14.078261,13.647577,14.420091,13.958848,13.802691
Cerveza Lata 355ml,40.356923,41.654434,41.347826,40.182390,40.737805
Cloro Blanqueador 1L,3.782313,3.382550,3.777070,3.673759,3.575949


In [23]:
# Producto × día de la semana
por_dia = eda_df.pivot_table(
    index="nombre_producto", columns="dia_semana",
    values="unidades_vendidas", aggfunc="mean", observed=True, fill_value=0,
)
por_dia

dia_semana,Lunes,Martes,Miércoles,Jueves,Viernes,Sábado,Domingo
nombre_producto,,,,,,,
Aceite Vegetal 1L,14.684211,13.531646,12.947059,14.142077,13.942675,13.216867,13.829545
Agua Mineral 1L,41.127193,41.383929,40.035714,40.965066,38.930435,38.883929,41.415254
Agua Sabor Limón 600ml,15.581081,15.048485,14.899329,15.755952,15.260606,15.474684,15.744048
Arroz Largo 1kg,39.312217,38.978723,39.062500,37.376682,38.444934,38.425439,38.353712
Atún Lata 140g,38.039648,38.144068,38.872807,37.907080,37.930131,37.581197,37.600877
Azúcar Blanca 1kg,14.810127,13.968354,14.023529,14.000000,14.063291,13.634731,14.404624
Café Molido 250g,14.208791,14.271523,13.961783,13.605096,14.184713,14.011976,13.614035
Cerveza Lata 355ml,40.911111,41.004167,41.053571,40.817391,39.808889,40.856540,41.510460
Cloro Blanqueador 1L,3.595745,3.452830,3.584158,3.773109,3.766990,3.578947,3.695652


In [24]:
# 3. Producto × mes
por_mes = eda_df.pivot_table(
    index="nombre_producto", columns="mes",
    values="unidades_vendidas", aggfunc="mean", observed=True, fill_value=0,
)
por_mes

mes,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
nombre_producto,,,,,,,,,,,,
Aceite Vegetal 1L,16.657407,13.511364,12.750000,12.747475,13.641509,11.152174,11.068627,13.392157,13.260417,13.948980,15.781250,17.106383
Agua Mineral 1L,42.551471,36.081967,32.492647,34.104478,38.558824,45.205882,49.447761,48.456000,36.765625,36.445255,36.305970,48.299270
Agua Sabor Limón 600ml,16.741935,13.344828,11.747253,12.187500,14.130435,16.793814,20.844037,20.670455,14.195402,13.615385,13.063158,16.547368
Arroz Largo 1kg,45.447761,36.932773,37.818841,37.975806,36.553030,33.611940,31.181818,39.277372,35.969697,38.000000,41.960630,47.194444
Atún Lata 140g,43.082090,37.637931,37.272727,36.676471,37.078014,32.503759,31.572464,36.428571,36.029851,36.798561,41.542636,49.727941
Azúcar Blanca 1kg,15.878788,13.300000,13.250000,14.230769,13.607843,11.954545,11.099010,13.957895,14.292683,14.153846,15.653061,18.010309
Café Molido 250g,16.140000,14.071429,14.163265,13.152941,13.505618,11.539326,11.372549,13.554455,13.954545,13.633028,14.950000,17.474227
Cerveza Lata 355ml,42.643357,39.211382,32.862319,33.077519,37.078014,45.365854,50.790210,50.884892,36.907692,36.914286,39.269231,44.347518
Cloro Blanqueador 1L,4.678571,3.578947,3.449275,4.000000,3.260870,3.300000,3.461538,3.281250,3.593750,3.478261,3.819672,3.835443
